In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Chuẩn hóa ảnh màu về kích thước 32x32 mặc định của CIFAR-10
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_base)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_base)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 0, correct / total

100%|██████████| 170M/170M [29:32<00:00, 96.2kB/s]


Câu 1

In [5]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Khớp shape: 32 kênh * kích thước ảnh 8x8 còn lại sau pool
        self.fc1 = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

num_epochs = 10
model = CIFAR_CNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    _, test_acc = evaluate(model, test_loader)
    print(f'Epoch {epoch+1}: Test Accuracy = {test_acc:.4f}')

Epoch 1: Test Accuracy = 0.5472
Epoch 2: Test Accuracy = 0.6354
Epoch 3: Test Accuracy = 0.6585
Epoch 4: Test Accuracy = 0.6727
Epoch 5: Test Accuracy = 0.6619
Epoch 6: Test Accuracy = 0.6736
Epoch 7: Test Accuracy = 0.6864
Epoch 8: Test Accuracy = 0.6904
Epoch 9: Test Accuracy = 0.6757
Epoch 10: Test Accuracy = 0.6916


Câu 2

In [6]:
class CIFAR_CNN_3Layers(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Khớp shape chuẩn: 64 kênh * ảnh kích thước 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x))) # 32x32 -> 16x16
        x = self.pool(torch.relu(self.conv2(x))) # 16x16 -> 8x8
        x = self.pool(torch.relu(self.conv3(x))) # 8x8 -> 4x4
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

model_3l = CIFAR_CNN_3Layers().to(device)
print(model_3l)

data, _ = next(iter(train_loader))
output = model_3l(data.to(device))
print(f"\nKích thước đầu ra chuẩn không lỗi: {output.shape}")

CIFAR_CNN_3Layers(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1024, out_features=10, bias=True)
)

Kích thước đầu ra chuẩn không lỗi: torch.Size([64, 10])


Câu 3

In [ ]:
lrs = [0.001, 0.01, 0.1]
plt.figure(figsize=(10, 6))

for lr in lrs:
    print(f"Đang chạy lr={lr}...")
    model_lr = CIFAR_CNN().to(device)
    optimizer = optim.SGD(model_lr.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    loss_history = []
    for epoch in range(5):
        model_lr.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_lr(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        loss_history.append(running_loss / len(train_loader))

    plt.plot(range(1, 6), loss_history, marker='o', label=f'lr = {lr}')

plt.title('So sánh Loss với các Learning Rate khác nhau')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.legend()
plt.grid(True)
plt.show()

Đang chạy lr=0.001...
Đang chạy lr=0.01...
Đang chạy lr=0.1...


Câu 4

In [ ]:
model.eval()
data, _ = next(iter(test_loader))
img = data[0:1].to(device)

with torch.no_grad():
    h1 = model.pool(torch.relu(model.conv1(img)))
    h2 = torch.relu(model.conv2(h1))

fmap_conv2 = h2[0, 0].cpu().numpy()
plt.figure(figsize=(6, 3))

plt.subplot(1, 2, 1)
orig_img = img.cpu().squeeze().permute(1, 2, 0).numpy()
orig_img = (orig_img * 0.5) + 0.5
plt.imshow(orig_img)
plt.title('Ảnh gốc')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(fmap_conv2, cmap='gray')
plt.title('Feature Map Conv2')
plt.axis('off')

plt.show()

Câu 5

In [ ]:
train_transform_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_loader_aug = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_transform_aug),
    batch_size=64, shuffle=True
)

class CIFAR_CNN_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=0.25)
        self.fc1 = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc1(x)
        return x

model_final = CIFAR_CNN_Dropout().to(device)
optimizer = optim.SGD(model_final.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

print("Bắt đầu chạy câu 5...")
for epoch in range(10):
    model_final.train()
    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_final(images), labels)
        loss.backward()
        optimizer.step()

    model_final.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_final(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    print(f"Epoch {epoch+1}/10 | Test Accuracy: {correct/total:.4f}")